# V3: extension feature preparation

Reuses the V3 CLIN cohort, outcome and fixed split; processes only the development set and does not read the held-out labels.

Temporal trajectories: 0–24 hours after ICU admission, labs from the same hadm_id and vital signs from the same stay_id; last-minus-first difference, slope per hour, SD, mean of the last 12 hours minus the first 12 hours, and number of measurements. Measurements at the same time point are averaged first. With fewer than two time points the slope and difference are missing; missingness does not remove the patient.

ICD documents: only hospitalizations that ended before the index hospitalization began are included; the ICD9/10 version, the exact code and the three-character category are all kept. Patients with no prior diagnoses are kept with an empty document. This step only prepares the documents; TF-IDF/SVD is not fitted in advance.

The V3 static columns keep the original −6 to 24 hour window, so the trajectories add dynamic information rather than redefining the original baseline. Descriptive missing rates are only for understanding the data; the later missing-rate screening is done within the training folds.

The main analysis inherits the V3 20% variable missingness rule and the exemption of the 8 white-cell differential variables. Trajectory variables are not exempt; the number of measurements is an indicator of testing intensity and should not be interpreted as a causal risk factor.

**balanced simplified-compute version**: fixed base-model parameters, full keeps the 5×10 outer validation; LASSO inner 5 folds/30 λ, stacking inner 3 folds. Uses a cache separate from the parent strict version; the old predictions of the first 6 folds are not mixed in. No real patient model of the new version has been run yet.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the project from the extension directory or the notebooks subdirectory, to avoid hard-coding a Windows drive letter.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dn_extensions" / "settings.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("start from dn_mortality_repro_V3_extensions or its notebooks subdirectory.")
sys.path.insert(0, str(ROOT))
print("extension directory:", ROOT)


In [ ]:
# By default only show the description and do not scan the database. Set to True to prepare features.
RUN_FEATURE_PREPARATION = False
if RUN_FEATURE_PREPARATION:
    from dn_extensions.features import build_features
    from dn_extensions.reporting import update_preparation_notebook
    manifest = build_features()
    update_preparation_notebook(manifest)
else:
    print("balanced shares the features prepared by the parent directory, no need to rerun prepare; if features are missing, run prepare in the parent directory.")


## Experiment status and conclusions (updated automatically)

The simplified version has not run any real patient experiment yet. The first 6 folds of the original strict version are saved and not mixed into this version.